In [ ]:
# Installation de TabPFN (Colab)
# Note: TabPFN requires scikit-learn < 1.6 due to reliance on private utils
%pip install -q tabpfn "scikit-learn<1.6.0"

In [ ]:
import pandas as pd
import numpy as np
import time
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report
from tabpfn import TabPFNClassifier

SEED = 42
np.random.seed(SEED)
print('Libraries loaded.')

## 1. Monter Google Drive et charger les données

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ⚠️ ADAPTER CE CHEMIN selon ton Drive
DATA_DIR = '/content/drive/My Drive/QRT-ChallengeENS/Data'

# Charger X_train_sample et y_train_sample
X_raw = pd.read_csv(f'{DATA_DIR}/X_train_sample.csv')
y_raw = pd.read_csv(f'{DATA_DIR}/y_train_sample.csv')

df_train = X_raw.merge(y_raw, on='ROW_ID')
print(f'Train Sample loaded : {df_train.shape[0]:,} lignes × {df_train.shape[1]} colonnes')

## 2. Feature Engineering & Normalisation

Définition des fonctions pour assurer la cohérence Train/Test.

In [ ]:
def feature_engineering(df_in):
    """
    Applique le Feature Engineering sur une copie du dataframe.
    """
    df = df_in.copy()
    
    # Base features lists
    ret_cols = [f'RET_{i}' for i in range(1, 21)]
    
    # 1. Interaction
    df['RET_1_TURNOVER'] = df['RET_1'] * df['MEDIAN_DAILY_TURNOVER']
    
    # 2. Rolling statistics
    df['RET_MEAN_5'] = df[[f'RET_{i}' for i in range(1, 6)]].mean(axis=1)
    df['RET_MEAN_20'] = df[ret_cols].mean(axis=1)
    df['RET_STD_5'] = df[[f'RET_{i}' for i in range(1, 6)]].std(axis=1)
    df['RET_STD_20'] = df[ret_cols].std(axis=1)
    
    # 3. Cumulative return
    df['RET_CUM_5'] = df[[f'RET_{i}' for i in range(1, 6)]].sum(axis=1)
    df['RET_CUM_20'] = df[ret_cols].sum(axis=1)
    
    # 4. Momentum indicators
    df['RET_POSITIVE_COUNT_5'] = (df[[f'RET_{i}' for i in range(1, 6)]] > 0).sum(axis=1)
    df['RET_POSITIVE_COUNT_20'] = (df[ret_cols] > 0).sum(axis=1)
    
    # 5. Recent vs Old
    df['RET_RECENT_VS_OLD'] = df['RET_MEAN_5'] - df[[f'RET_{i}' for i in range(16, 21)]].mean(axis=1)
    
    return df

def normalize_by_group(df, scalers=None):
    """
    Normalisation par groupe.
    - Si scalers est None (Train) : on fit et on retourne (df_scaled, scalers)
    - Si scalers est fourni (Test) : on transform avec les scalers existants
    """
    df_scaled = df.copy()
    feature_cols = [c for c in df_scaled.columns if c not in ['ROW_ID', 'TS', 'ALLOCATION', 'target', 'target_SIGN', 'GROUP']]
    
    if scalers is None:
        # Mode Train : Fit
        scalers = {}
        for grp in df_scaled['GROUP'].unique():
            mask = df_scaled['GROUP'] == grp
            scaler = StandardScaler()
            df_scaled.loc[mask, feature_cols] = scaler.fit_transform(df_scaled.loc[mask, feature_cols])
            scalers[grp] = scaler
        return df_scaled, scalers
    else:
        # Mode Test : Transform
        for grp in df_scaled['GROUP'].unique():
            mask = df_scaled['GROUP'] == grp
            if grp in scalers:
                scaler = scalers[grp]
                df_scaled.loc[mask, feature_cols] = scaler.transform(df_scaled.loc[mask, feature_cols])
            else:
                # Fallback pour groupes inconnus (rare/impossible ici car le sample a tous les groupes)
                scaler = StandardScaler()
                df_scaled.loc[mask, feature_cols] = scaler.fit_transform(df_scaled.loc[mask, feature_cols])
        return df_scaled

print('Fonctions FE définies.')

In [ ]:
# 1. Application FE sur Train
df_fe = feature_engineering(df_train)

# 2. Target
df_fe['target_SIGN'] = (df_fe['target'] > 0).astype(int)

# 3. Normalisation (Fit)
df_norm, train_scalers = normalize_by_group(df_fe, scalers=None)

# 4. Préparation X, y
exclude_cols = ['ROW_ID', 'TS', 'ALLOCATION', 'target', 'target_SIGN']
feature_cols = [c for c in df_norm.columns if c not in exclude_cols]

X = df_norm[feature_cols].fillna(0).values.astype(np.float32)
y = df_norm['target_SIGN'].values

# 5. Split Train/Val (80/20)
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)

print(f'Train shape : {X_train.shape}')
print(f'Val shape   : {X_val.shape}')
print(f'Features ({len(feature_cols)}) : {feature_cols}')

## 3. Entraînement TabPFN

In [ ]:
print(f'=== TabPFN Training ===')
t0 = time.time()

# ignore_pretraining_limits=True pour large datasets
clf_full = TabPFNClassifier(ignore_pretraining_limits=True)
clf_full.fit(X_train, y_train)

y_pred_val = clf_full.predict(X_val)
y_prob_val = clf_full.predict_proba(X_val)[:, 1]

elapsed = time.time() - t0
acc_val = accuracy_score(y_val, y_pred_val)
auc_val = roc_auc_score(y_val, y_prob_val)

print(f'Val Accuracy : {acc_val:.4f}')
print(f'Val ROC AUC  : {auc_val:.4f}')
print(f'Temps        : {elapsed:.1f}s')

print('\nInternal Val Report:')
print(classification_report(y_val, y_pred_val))

## 4. Générer la Submission pour QRT

**Important** : On applique exactement le même FE et la même normalisation (avec les scalers du train).

In [ ]:
print('=== Génération de la submission ===')

# --- Safety Checks ---
if 'DATA_DIR' not in locals():
    DATA_DIR = '/content/drive/My Drive/QRT-ChallengeENS/Data'
    print(f'⚠️ DATA_DIR not defined, using default: {DATA_DIR}')

if 'clf_full' not in locals():
    raise NameError("⚠️ Model 'clf_full' not found. Please run the Training cells first!")
# ---------------------

# 1. Charger le test set
X_test_raw = pd.read_csv(f'{DATA_DIR}/X_test.csv')
print(f'Test set loaded : {X_test_raw.shape}')
test_row_ids = X_test_raw['ROW_ID']

# 2. Appliquer le MÊME Feature Engineering
X_test_fe = feature_engineering(X_test_raw)

# 3. Appliquer la MÊME Normalisation (Transform uniquement)
X_test_norm = normalize_by_group(X_test_fe, scalers=train_scalers)

# 4. Sélectionner les mêmes colonnes
X_test = X_test_norm[feature_cols].fillna(0).values.astype(np.float32)

# 5. Prédiction
print('Prédiction en cours...')
y_pred_test = clf_full.predict(X_test)

print(f'Prédictions statistiques :')
print(f'  Up (1)   : {(y_pred_test == 1).sum()} ({(y_pred_test == 1).mean()*100:.1f}%)')
print(f'  Down (0) : {(y_pred_test == 0).sum()} ({(y_pred_test == 0).mean()*100:.1f}%)')

# 6. Sauvegarde
submission = pd.DataFrame({
    'ROW_ID': test_row_ids,
    'prediction': y_pred_test
})

submission.to_csv('submission_tabpfn_fe.csv', index=False)
print('✅ Fichier submission_tabpfn_fe.csv créé.')

# 7. Téléchargement automatique
from google.colab import files
files.download('submission_tabpfn_fe.csv')